# Condition Authoring & Testing

This notebook demonstrates how to:

1. Write and test condition expressions against real data
2. Debug why a step was skipped or executed
3. Build complex multi-clause conditions
4. Validate conditions across multiple input scenarios
5. Apply tested conditions to pipeline steps

Conditions control whether a pipeline step executes or is skipped.
Getting them right **before deployment** prevents wasted runs.

In [ ]:
from agent.notebook import Session

s = Session()

# Show supported syntax
s.conditions.show_operators()

## 1. Basic Condition Testing

Test simple comparisons against context data.

In [ ]:
# Simulate a video file's metadata as context
video_context = {
    "resolution": 1080,
    "width": 1920,
    "height": 1080,
    "duration": 185,       # seconds
    "fps": 30,
    "codec": "h264",
    "has_audio": True,
    "file_size": 250_000_000,  # 250MB
}

# Test individual conditions
print("Test results:")
print(f"  resolution != 720   -> {s.conditions.test('resolution != 720', video_context)}")
print(f"  duration > 60       -> {s.conditions.test('duration > 60', video_context)}")
print(f"  fps >= 24           -> {s.conditions.test('fps >= 24', video_context)}")
print(f"  resolution == 4096  -> {s.conditions.test('resolution == 4096', video_context)}")

## 2. Existence Checks with `has()`

Test whether an artifact or context key exists before using it.

In [ ]:
# Check if previous steps produced expected artifacts
pipeline_context = {
    "video_path": "/data/input.mp4",
    "timecodes": [1.5, 4.2, 8.0, 12.3],
    "clips": ["/tmp/clip_001.mp4", "/tmp/clip_002.mp4"],
    "error": None,  # None = not present
    "face_count": 3,
}

tests = [
    'has(timecodes)',       # True: list with items
    'has(clips)',           # True: list with items
    'has(error)',           # False: value is None
    'has(nonexistent)',     # False: key doesn't exist
    'not has(error)',       # True: negated None check
    'has(face_count)',      # True: int value exists
]

for expr, result in s.conditions.test_batch(tests, pipeline_context):
    status = 'PASS' if result else 'FAIL'
    print(f"  {status}  {expr}")

## 3. Combined Conditions with `and` / `or`

Build multi-clause conditions for complex logic.

In [ ]:
# Use case: Only run assembly if we have clips AND no errors
s.conditions.explain(
    "has(clips) and not has(error)",
    pipeline_context,
)

In [ ]:
# Use case: Run resize only if not already 720p AND file is large
s.conditions.explain(
    "resolution != 720 and file_size > 100000000",
    video_context,
)

In [ ]:
# Use case: Run GPU processing if faces found OR duration is short
# (short videos can afford GPU even without face pre-check)
ctx_with_faces = {"face_count": 5, "duration": 300}
ctx_short = {"face_count": 0, "duration": 10}
ctx_long_no_faces = {"face_count": 0, "duration": 300}

condition = "face_count > 0 or duration < 30"

print(f"Condition: {condition!r}\n")
print(f"  With faces, long:   {s.conditions.test(condition, ctx_with_faces)}")
print(f"  No faces, short:    {s.conditions.test(condition, ctx_short)}")
print(f"  No faces, long:     {s.conditions.test(condition, ctx_long_no_faces)}")

## 4. Scenario Matrix Testing

Test a condition against many input variations to ensure it
behaves correctly across edge cases.

In [ ]:
# Resize condition: should fire when resolution is not 720p
resize_condition = "resolution != 720"

scenarios = [
    {"name": "4K video",      "ctx": {"resolution": 2160}, "expected": True},
    {"name": "1080p video",   "ctx": {"resolution": 1080}, "expected": True},
    {"name": "720p video",    "ctx": {"resolution": 720},  "expected": False},
    {"name": "480p video",    "ctx": {"resolution": 480},  "expected": True},
    {"name": "360p video",    "ctx": {"resolution": 360},  "expected": True},
]

print(f"Testing: {resize_condition!r}\n")
all_pass = True
for scenario in scenarios:
    actual = s.conditions.test(resize_condition, scenario["ctx"])
    match = actual == scenario["expected"]
    status = "OK" if match else "FAIL"
    if not match:
        all_pass = False
    print(f"  {status:4s}  {scenario['name']:<15s}  expected={scenario['expected']}  got={actual}")

print(f"\n{'All scenarios passed!' if all_pass else 'SOME SCENARIOS FAILED'}")

In [ ]:
# More complex scenario: assembly condition
assembly_condition = "has(clips) and not has(error)"

assembly_scenarios = [
    {"name": "Normal",       "ctx": {"clips": ["a.mp4"], "error": None},  "expected": True},
    {"name": "With error",   "ctx": {"clips": ["a.mp4"], "error": "fail"}, "expected": False},
    {"name": "No clips",     "ctx": {"clips": None, "error": None},        "expected": False},
    {"name": "Empty clips",  "ctx": {"error": None},                       "expected": False},
    {"name": "Both bad",     "ctx": {"error": "fail"},                     "expected": False},
]

print(f"Testing: {assembly_condition!r}\n")
for scenario in assembly_scenarios:
    actual = s.conditions.test(assembly_condition, scenario["ctx"])
    match = actual == scenario["expected"]
    status = "OK" if match else "FAIL"
    print(f"  {status:4s}  {scenario['name']:<15s}  expected={scenario['expected']}  got={actual}")

## 5. Apply Conditions to a Pipeline

Once conditions are validated, apply them to pipeline steps.

In [ ]:
# Build a pipeline with the tested conditions
pipeline, steps = s.compiler.from_steps(
    [
        {
            "name": "resize",
            "command_type": "ffmpeg",
            "command_template": "ffmpeg -i {video_path} -vf scale=1280:720 {resized_path}",
            "condition": "resolution != 720",  # validated above
        },
        {
            "name": "scene_detect",
            "command_type": "python",
            "command_template": "python scene_detect.py --input {resized_path}",
            "depends_on_names": ["resize"],
        },
        {
            "name": "face_filter",
            "command_type": "python",
            "command_template": "python facedetection.py --input {clip} --threshold 0.3",
            "depends_on_names": ["scene_detect"],
            "fan_out_on": "clips",
        },
        {
            "name": "assemble",
            "command_type": "ffmpeg",
            "command_template": "ffmpeg -f concat -i {clip_list} -c copy {output}",
            "depends_on_names": ["face_filter"],
            "condition": "has(clips) and not has(error)",  # validated above
        },
    ],
    name="Conditioned Pipeline",
    inputs={"video_path": "/data/input.mp4"},
)

# Show pipeline with conditions
s.pipelines.inspect(pipeline, steps)

In [ ]:
# Simulate: what would happen with a 720p input?
print("Simulation: 720p input video\n")
context_720p = {"resolution": 720, "clips": ["a.mp4", "b.mp4"], "error": None}

for step in steps:
    if step.condition:
        result = s.conditions.test(step.condition, context_720p)
        action = "EXECUTE" if result else "SKIP"
        print(f"  Step '{step.name}': {action}  (condition: {step.condition})")
    else:
        print(f"  Step '{step.name}': EXECUTE  (no condition)")

In [ ]:
# Simulate: what would happen with a 1080p input?
print("Simulation: 1080p input video\n")
context_1080p = {"resolution": 1080, "clips": ["a.mp4"], "error": None}

for step in steps:
    if step.condition:
        result = s.conditions.test(step.condition, context_1080p)
        action = "EXECUTE" if result else "SKIP"
        print(f"  Step '{step.name}': {action}  (condition: {step.condition})")
    else:
        print(f"  Step '{step.name}': EXECUTE  (no condition)")

In [ ]:
# Simulate: what if there's an error in processing?
print("Simulation: processing error occurred\n")
context_error = {"resolution": 1080, "clips": ["a.mp4"], "error": "GPU out of memory"}

for step in steps:
    if step.condition:
        result = s.conditions.test(step.condition, context_error)
        action = "EXECUTE" if result else "SKIP"
        print(f"  Step '{step.name}': {action}  (condition: {step.condition})")
    else:
        print(f"  Step '{step.name}': EXECUTE  (no condition)")

## 6. Save and Export

Once all conditions are validated, save the pipeline.

In [ ]:
# Save as template (persisted to DB)
template = s.pipelines.save_template(
    pipeline=pipeline,
    steps=steps,
    name="Conditioned Video Pipeline v1",
)
print(f"Saved template: {template.name} ({template.id})")

# Export for git
s.export.pipeline((pipeline, steps), "pipelines/conditioned_v1.json")

In [ ]:
s.close()